In [10]:
import itertools
import numpy as np
from sklearn.preprocessing import scale
from sklearn.utils import check_array, check_scalar

from lingam import DirectLiNGAM

class HighDimDirectLiNGAM(DirectLiNGAM):

    def __init__(self, K=4, J=3, **kwargs):
        super().__init__(**kwargs)

        self._K = check_scalar(K, "K", int, min_val=2,include_boundaries="neither")
        self._J = check_scalar(J, "J", int, min_val=1)


    def fit(self, X):
        """Fit the model to X.

        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            Training data, where ``n_samples`` is the number of samples
            and ``n_features`` is the number of features.

        Returns
        -------
        self : object
            Returns the instance itself.
        """
        # Check parameters
        Y = check_array(X, copy=True)

        n_features = Y.shape[1]

        # Causal discovery
        theta = []
        psi = set(range(n_features))
        # C[z][v]
        C = {}
        
        # 書き換えることのない固定値は大域変数みたいな扱いでもいいのかも。全部private変数に入れ込む。
        alpha = 0.5
        
        for z in range(n_features):
            Cz = {}
            stats = []
            g = []
            
            print("1")
            for v in psi:
                # search possible parents
                # XXX: 最初は親候補がないので常にCvは空？gはゼロ？とりあえず空とゼロで。
                print("2")
                Cv, gz = self._possible_parents(z, v, C, g, theta, self._J, alpha, psi, Y, self._K)
                print("3")
                stat = self._T(v, Cv, psi - set([v]), self._J, Y, self._K)
                print("4")
                
                Cz[v] = Cv
                g.append(gz)
                stats.append((stat, v))
            _, r = min(stats)
            theta.append(r)
            psi.remove(r)
            C[z] = Cz
            print("5")

        # prune?
        self._causal_order = theta
        #return self._estimate_adjacency_matrix(X, prior_knowledge=self._Aknw)
        self._adjacency_matrix = np.zeros((len(Y.T), len(Y.T)))
        return self

    def _tau(self, v, u, C, Y, K):
        sigma = np.cov(Y.T, bias=True)
        sigma_cc = sigma[C, :][:, C]
        Bvc = np.linalg.pinv(sigma_cc) * sigma[C, v]
        if C is None:
            Yvi_C = Y[:, v]
        else:
            Yvi_C = Y[:, v] - Y[:, C] @ Bvc
        
        # XXX: 期待値は平均値？
        tau = np.mean(Yvi_C ** (K - 1) @ Y[:, u]) * np.mean(Yvi_C ** 2) \
                                - np.mean(Yvi_C ** K) * np.mean(Yvi_C @ Y[:, u])
        return tau

    def _possible_parents(self, z, v, C, g, theta, J, alpha, psi, Y, K):
        # XXX: 前ステップのルートrを参照するが最初はないので、親候補なし。
        if len(theta) == 0:
            # 親なし、gはゼロ(どのみちCが空で候補がつくれない)
            return set(), 0
        
        # gz
        r = theta[-1]
        # XXX: C(z)_vを決めるために、gzを決めたいのに、C(z)_rが必要になるのでは鶏と卵みたいになる。
        # XXX: いったんC[z-1][r]にした。
        print(r, C[z - 1][r], psi, J, Y, K)
        x = self._T(r, C[z - 1][r], psi, J, Y, K)
        print(alpha, x)
        gz = max([*g, alpha * self._T(r, C[z - 1][r], psi, J, Y, K)])

        Cv = set()
        for p in C[z - 1][v]:
            # Dv
            Dv = set()
            for d in range(z):
                if v not in C[d]:
                    continue

                # XXX: Dvに空集合は入るのか？
                for i in range(1, J + 1):
                    comb = itertools.combinations(C[d][v] - set([p]), i)
                    Dv |= set(comb)
            
            # 判定
            taus = []
            for C in Dv:
                tau = self._tau(v, p, C, Y, K)
                taus.append(tau)
                
            if min(taus) <= gz:
                continue
            
            Cv |= set([p])
            
        Cv |= set(theta)
        return Cv, gz

    def _T(self, v, V1, V2, J, Y, K, is_T1=True):
        # XXX: V1が空だと空集合の要素を探すことになってしまう。
        # XXX: そこは無視してCは空としてtauの計算はできる。
        # XXX: tauの計算時のYui.Cについて、Cが空のとき残差はデータと同一。
        if len(V1) == 0:
            # XXX: V1Jのループはない。V1Jが空集合なので消してみた。
            taus = []
            for u in V2:
                tau = self._tau(v, u, None, Y, K)
                taus.append((tau, u))
            tau, C = max(taus)
            return 
        
        if J <= len(V1):
            V1J = itertools.combinations(V1, J)
        else:
            V1J = [tuple(V1)]
        
        if is_T1:
            max_taus = []
            for C in V1J:
                taus = []
                for u in V2:
                    tau = self._tau(v, u, C, Y, K)
                    tau = abs(tau ** K)
                    taus.append(tau)
                max_tau = max(taus)
                max_taus.append((max_tau, C))
            _, C = min(max_taus)
        else:
            min_taus = []
            for u in V2:
                taus = []
                for C in V1J:
                    tau = self._tau(v, u, C, Y, K)
                    tau = abs(tau ** K)
                    taus.append(tau)
                min_tau = min(taus)
                min_taus.append((min_tau, C))
            _, C = max(min_taus)
        
        return C

    
    
import lingam

m = np.array([
    [ 0.000,  0.000,  0.000,  0.895,  0.000,  0.000],
    [ 0.565,  0.000,  0.377,  0.000,  0.000,  0.000],
    [ 0.000,  0.000,  0.000,  0.895,  0.000,  0.000],
    [ 0.000,  0.000,  0.000,  0.000,  0.000,  0.000],
    [ 0.991,  0.000, -0.124,  0.000,  0.000,  0.000],
    [ 0.895,  0.000,  0.000,  0.000,  0.000,  0.000]
])

sample_size = 10000

error_vars = [0.2, 0.2, 0.2, 1.0, 0.2, 0.2]
params = [0.5 * np.sqrt(12 * v) for v in error_vars]

generate_error = lambda p: np.random.uniform(-p, p, size=sample_size)
e = np.array([generate_error(p) for p in params])

X = np.linalg.pinv(np.eye(len(m)) - m) @ e
X = X.T

model = HighDimDirectLiNGAM()
model.fit(X)
model.causal_order_

1
2
3
4
2
3
4
2
3
4
2
3
4
2
3
4
2
3
4
5
1
2
0 set() {1, 2, 3, 4, 5} 3 [[-0.36571953 -0.7518875   0.4237405   0.17822482 -0.21438022  0.10062065]
 [ 0.89431693  1.21367093  0.35481416  1.21206483  1.47322551  0.86735823]
 [-0.67490085 -0.8585424  -1.85707779 -1.30727346 -0.49051457 -0.40183125]
 ...
 [ 0.79763703  0.09422366  0.36462225  1.14877765  0.96012652  1.13076357]
 [-1.77488998 -1.05557928 -1.60134298 -1.28170554 -2.02924672 -1.56262679]
 [-1.34258827 -1.76880542 -1.16366646 -1.41617645 -1.33872367 -0.63788693]] 4
0.5 set()


TypeError: unsupported operand type(s) for *: 'float' and 'set'

In [ ]:
C = set({})
C |= set({1})
C

In [ ]:
list(itertools.combinations([1,2,3], 1))